In [ ]:
import json
import numpy as np

def audit_json_quality(filepath):
    print(f"🔍 Memulai audit integritas untuk: {filepath}")
    
    try:
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        total_records = len(data)
        if total_records == 0:
            print("❌ File JSON kosong!")
            return

        records_with_padding = 0
        records_with_nan = 0
        records_with_flatline = 0 
        
        for key, record in data.items():
            # Mengambil komponen Z (asumsi ada di setiap record)
            z_signal = record.get('Z', [])
            
            # 1. Cek NaN / Inf
            if any(np.isnan(z_signal)) or any(np.isinf(z_signal)):
                records_with_nan += 1
                
            # 2. Cek Zero-Padding (cek apakah 10 titik terakhir adalah nol semua)
            # Indikasi kuat sinyal terpotong atau padding buatan
            if len(z_signal) > 10 and all(abs(val) < 1e-6 for val in z_signal[-10:]):
                records_with_padding += 1
            
            # 3. Cek Flatline (Variansi sangat kecil mendekati nol = sinyal mati)
            if len(z_signal) > 0 and np.std(z_signal) < 1e-6:
                records_with_flatline += 1
                
        print("="*60)
        print("LAPORAN HASIL AUDIT")
        print("="*60)
        print(f"Total Record           : {total_records}")
        print(f"Record NaN/Inf         : {records_with_nan}")
        print(f"Record Flatline        : {records_with_flatline}")
        print(f"Record Zero-Padding    : {records_with_padding} ({records_with_padding/total_records*100:.2f}%)")
        print("="*60)
        
        if records_with_padding > 0 or records_with_nan > 0:
            print("⚠️ REKOMENDASI: Ada record yang cacat. Periksa pipeline ekstraksi")
        else:
            print("✅ DATA CLEAN: Seluruh record bebas dari artefak numerik.")

    except Exception as e:
        print(f"❌ Terjadi kesalahan saat audit: {e}")

if __name__ == "__main__":
    # Path file Bapak
    filepath = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_final.json'
    audit_json_quality(filepath)

🔍 Memulai audit integritas untuk: /Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_3c_4_final.json
LAPORAN HASIL AUDIT
Total Record           : 1964
Record NaN/Inf         : 0
Record Flatline        : 4
Record Zero-Padding    : 5 (0.25%)
⚠️ REKOMENDASI: Ada record yang cacat. Periksa pipeline ekstraksi Bapak.
